In [ ]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

In [ ]:
def test_data_loader():
    print("==================================================")
    print("🚀 QuantDataLoader 테스트를 시작합니다...")
    print("==================================================\n")
    
    # 1. 로더 인스턴스 생성
    try:
        print("[테스트 1] 로더 인스턴스화 및 환경변수 확인")
        loader = QuantDataLoader(use_cache=True)
        print("✅ 성공: DART API 키 및 로더 초기화 완료!\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")
        return

    # 2. 유니버스 로드 테스트 (Point-in-Time)
    test_date = date(2023, 7, 24)
    print(f"[테스트 2] KOSPI 유니버스 데이터 로드 ({test_date})")
    try:
        universe_df = loader.get_kospi_universe(test_date)
        print(f"✅ 성공: 총 {len(universe_df)}개 종목 로드 완료!")
        print("-" * 50)
        display(universe_df.head())
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    # 3. DART 재무제표 파싱 테스트
    ticker_to_test = '005930'
    target_year = 2023
    print(f"[테스트 3] {ticker_to_test} {target_year}년 사업보고서(11011) 파싱")
    try:
        financials = loader.parse_standardized_financials(ticker_to_test, target_year, '11011')
        print(f"✅ 성공: 재무 데이터 표준화 완료!")
        print("-" * 50)
        for key, value in financials.items():
            if pd.isna(value):
                print(f"{key:>20} : NaN")
            else:
                print(f"{key:>20} : {value:,.0f}")
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    print("==================================================")
    print("🎯 모든 테스트가 종료되었습니다.")
    print("==================================================")

# 테스트 실행
test_data_loader()

In [ ]:
# config.json 파라미터 불러오기
try:
    with open('config.json', 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    target_market = config['strategy_params']['market'] # 예: 'KOSPI'
    top_n = config['strategy_params']['top_n_mcap']
except FileNotFoundError:
    print("⚠️ config.json 파일이 없습니다. 기본값으로 진행합니다.")
    target_market = 'KOSPI'
    top_n = 10

print(f"🔍 {target_market} 시장 데이터를 불러오는 중...\n")

# FinanceDataReader를 통한 KRX 전종목 리스팅 조회
df_krx = fdr.StockListing('KRX')

# 지정한 시장 필터링 및 시가총액(MarCap) 기준 정렬
top_mcap_df = df_krx[df_krx['Market'] == target_market].sort_values(by='Marcap', ascending=False).head(top_n)

# 보기 좋게 컬럼명 정리
top_mcap_df = top_mcap_df[['Code', 'Name', 'Close', 'Marcap', 'Stocks']].rename(
    columns={
        'Code': '종목코드',
        'Name': '종목명',
        'Close': '종가',
        'Marcap': '시가총액',
        'Stocks': '상장주식수'
    }
)

print("✅ 정상적으로 데이터를 불러왔습니다!")
display(top_mcap_df)

In [ ]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

def verify_new_features():
    print("==================================================")
    print("🚀 QuantDataLoader 신규 기능 검증을 시작합니다...")
    print("==================================================\n")
    
    try:
        loader = QuantDataLoader(use_cache=True)
    except Exception as e:
        print(f"❌ 초기화 실패: {e}")
        return

    # ---------------------------------------------------------
    # 검증 1: 시계열 주가/거래량 데이터 (OHLCV) 및 캐싱
    # ---------------------------------------------------------
    print("[검증 1] get_historical_ohlcv 작동 확인")
    ticker = '005930'
    start = date(2025, 1, 1)
    end = date(2025, 6, 30)
    
    try:
        ohlcv_df = loader.get_historical_ohlcv(ticker, start, end)
        if ohlcv_df is not None and not ohlcv_df.empty:
            print(f"✅ 성공: {start} ~ {end} 시계열 데이터 {len(ohlcv_df)}일치 로드 완료")
            print(ohlcv_df[['Close', 'Volume']].head(3).to_string())
        else:
            print("❌ 실패: 데이터가 비어 있습니다.")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 2: 확장된 계정과목 매핑 확인 (자산, 부채, 자본 등)
    # ---------------------------------------------------------
    print("[검증 2] 확장된 재무제표 계정 파싱 확인 (2025년 사업보고서)")
    try:
        fin_annual = loader.parse_standardized_financials(ticker, 2025, '11011')
        keys_to_check = ['total_assets', 'total_liabilities', 'total_equity', 'interest_expense']
        
        missing = [k for k in keys_to_check if pd.isna(fin_annual.get(k, float('nan')))]
        if not missing:
            print("✅ 성공: 자산/부채/자본/이자비용 모두 정상 파싱 완료")
            for k in keys_to_check:
                print(f"   - {k}: {fin_annual[k]:,.0f}")
        else:
            print(f"⚠️ 주의: 다음 계정 누락 (정상일 수도 있음) -> {missing}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 3: 분기 단독값 차분(Isolation) 로직 확인
    # ---------------------------------------------------------
    print("[검증 3] get_isolated_quarterly_financials 차분 로직 확인")
    try:
        # 1분기(누적)와 2분기(차분)의 매출액 비교
        q1_data = loader.get_isolated_quarterly_financials(ticker, 2025, 1)
        q2_isolated = loader.get_isolated_quarterly_financials(ticker, 2025, 2)
        
        print("✅ 성공: 1분기 및 2분기(단독) 데이터 추출 완료")
        print(f"   - 1Q 매출액 (누적=단독) : {q1_data.get('revenue', 0):,.0f}")
        print(f"   - 2Q 매출액 (차분 적용) : {q2_isolated.get('revenue', 0):,.0f}")
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("-" * 50 + "\n")

    # ---------------------------------------------------------
    # 검증 4: DART API Rate Limit 카운터 작동 확인
    # ---------------------------------------------------------
    print("[검증 4] DART API 카운터 및 Rate Limit 방어벽 확인")
    try:
        current_calls = loader.dart_call_count
        print(f"✅ 성공: 현재 세션 API 호출 횟수 정상 트래킹 중 -> {current_calls}회")
        
        # 임의로 한도를 초과시켜 방어벽 테스트
        loader.dart_daily_limit = current_calls  
        
        # 💡 API를 강제로 호출하도록 일시적으로 캐시 기능 비활성화
        loader.use_cache = False 
        
        try:
            loader.get_financial_statements(ticker, 2024, '11011')
            print("❌ 실패: 한도 초과 상황에서 Exception이 발생하지 않고 통과됨!")
        except Exception as limit_err:
            print(f"✅ 성공: 방어벽 정상 작동 확인 -> {limit_err}")
            
    except Exception as e:
        print(f"❌ 실패: {e}")
    print("==================================================\n")


if __name__ == "__main__":
    verify_new_features()

In [ ]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

print("==================================================")
print("🚀 [검증] 공시 시차(Disclosure Lag) 및 미래참조 방지 테스트")
print("==================================================\n")

loader = QuantDataLoader(use_cache=True)
ticker = '005930' # 삼성전자
year = 2025
quarter = 1

# ---------------------------------------------------------
# 테스트 1: 공시 이전 시점 (Look-ahead bias 발생 가능 시점)
# 1분기 보고서 제출 마감일(보통 5월 15일) 이전인 '4월 30일' 기준
# ---------------------------------------------------------
base_date_before_release = date(2025, 4, 30)
print(f"▶️ 테스트 1: 기준일 = {base_date_before_release} (1Q 실적발표 전)")
data_before = loader.get_isolated_quarterly_financials(ticker, year, quarter, base_date=base_date_before_release)

if pd.isna(data_before.get('revenue')):
    print("✅ 성공: 아직 공시되지 않은 미래의 데이터를 정확히 차단하여 NaN을 반환했습니다.")
else:
    print(f"❌ 실패: 공시 전인데 미래 데이터를 가져왔습니다! 매출액: {data_before.get('revenue')}")

print("-" * 50)

# ---------------------------------------------------------
# 테스트 2: 공시 이후 시점 (정상적인 데이터 수집)
# 1분기 보고서 제출 마감일 이후인 '6월 1일' 기준
# ---------------------------------------------------------
base_date_after_release = date(2025, 6, 1)
print(f"\n▶️ 테스트 2: 기준일 = {base_date_after_release} (1Q 실적발표 후)")
data_after = loader.get_isolated_quarterly_financials(ticker, year, quarter, base_date=base_date_after_release)

if pd.notna(data_after.get('revenue')):
    print(f"✅ 성공: 공시가 완료된 데이터를 정상적으로 불러왔습니다. 매출액: {data_after.get('revenue'):,.0f}")
else:
    print("❌ 실패: 공시 이후임에도 데이터를 가져오지 못했습니다.")
print("\n==================================================")

In [ ]:
from data.loader import QuantDataLoader
from datetime import date
import pprint

print("==================================================")
print("🔍 [디버깅] get_quarterly_financials_series 원자료 점검")
print("==================================================\n")

loader = QuantDataLoader()
base_date = date(2026, 7, 31)
test_ticker = '005930'  # 삼성전자

try:
    print(f"⏳ [{test_ticker}] 삼성전자 최근 6개 분기 원자료 조회 중...")
    q_series = loader.get_quarterly_financials_series(test_ticker, base_date, n_quarters=6)
    
    print(f"\n✅ 반환된 리스트 길이 (분기 수): {len(q_series)}")
    
    if len(q_series) == 0:
        print("❌ 빈 리스트가 반환되었습니다. DART API 호출 로직 자체를 점검해야 합니다.")
    else:
        for i, q_data in enumerate(q_series):
            # t=0이 가장 최근 분기, t=5가 5분기 전(직전분기의 전년동기)
            print(f"\n--------------------------------------------------")
            print(f"📅 [t-{i} 분기 데이터]")
            print(f"--------------------------------------------------")
            pprint.pprint(q_data, indent=2, width=80)
            
            # Stage 3에서 필수로 찾는 키값들이 있는지 체크
            required_keys = ['revenue', 'sga', 'gross_profit', 'inventory']
            missing_keys = [key for key in required_keys if key not in q_data or pd.isna(q_data.get(key))]
            if missing_keys:
                print(f"⚠️ 경고: Stage 3 필수 계정 누락 또는 NaN -> {missing_keys}")

except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date

# 구현해두신 모듈들을 임포트합니다.
# (경로나 클래스명은 실제 프로젝트 환경에 맞게 조정해 주세요)
from data.loader import QuantDataLoader
from stages.stage1_neglected_sector import NeglectedSectorScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 1: KOSPI 소외 섹터 발굴")
print("==================================================\n")

# 1. 기준일 설정 (현재 날짜 기준)
base_date = date(2026, 7, 31)

# 2. 파라미터 세팅 (config/params.yaml에서 불러오는 것을 모사)
stage1_params = {
    'stage1_return_weight': 0.5,
    'stage1_volume_weight': 0.5,
    'stage1_pass_ratio': 0.4  # 상위 40% 섹터 통과
}

try:
    # 3. 로더 초기화 및 실제 섹터 데이터 조달
    # (내부적으로 pykrx 등을 호출하여 데이터를 가져온다고 가정합니다)
    loader = QuantDataLoader()
    
    print("⏳ KOSPI 섹터 시계열 데이터 수집 중...")
    sector_metrics_df = loader.get_sector_metrics(base_date)
    
    if sector_metrics_df.empty:
        print("❌ 섹터 데이터를 불러오지 못했습니다. 로더의 구현 상태를 확인해 주세요.")
    else:
        # 4. Stage 1 스크리너 실행
        screener = NeglectedSectorScreener(params=stage1_params)
        passed_sectors = screener.run(sector_metrics_df)
        
        print(f"\n📊 [기준일: {base_date}] KOSPI 전체 섹터 데이터 (상위 5개)")
        display(sector_metrics_df.head())
        
        print(f"\n✅ [Stage 1 통과] 최종 소외 섹터 (Pass Ratio: {stage1_params['stage1_pass_ratio']*100}%)")
        display(passed_sectors)
        
        # 밸류트랩 경고 확인
        value_traps = passed_sectors[passed_sectors['is_value_trap_warning'] == True]
        if not value_traps.empty:
            print("\n⚠️ [주의] 다음 섹터는 통과되었으나 낙폭 가속(Value Trap) 위험이 감지되었습니다:")
            print(value_traps['sector'].tolist())

except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date

from data.loader import QuantDataLoader
from stages.stage2_sector_leaders import SectorLeaderScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 2: 섹터 내 우량주 탐색")
print("==================================================\n")

# 1. 기준일 설정 (현재 날짜 기준)
base_date = date(2026, 7, 31)

# 2. Stage 2 파라미터 세팅
stage2_params = {
    'roe_percentile_cutoff': 0.5,
    'roic_percentile_cutoff': 0.5,
    'op_margin_std_percentile_cutoff': 0.5,
    'op_margin_lookback_q': 8,
    'op_margin_min_quarters': 4
}

try:
    loader = QuantDataLoader()
    screener = SectorLeaderScreener(params=stage2_params)
    
    # 3. 테스트용 실제 종목 리스트 
    # (Stage 1을 통과한 종목들이라고 가정하고, 익숙한 KOSPI 종목들로 구성)
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '005380', '068270'],  # 삼성전자, SK하이닉스, 현대차, 셀트리온
        'sector': ['IT', 'IT', '자동차', '바이오'] 
    })
    
    print("⏳ DART API 실전 재무 데이터 수집 및 지표 계산 중...")
    print("(종목별로 과거 8개 분기 원자료를 가져와야 하므로 시간이 약간 소요될 수 있습니다.)\n")
    
    # 4. Stage 2 스크리너 실행
    passed_stage2_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 2 통과] 최종 섹터 우량주")
    display(passed_stage2_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date

from data.loader import QuantDataLoader
from stages.stage3_fundamental_improve import FundamentalImproveScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 3: 체질 개선 (Turnaround)")
print("==================================================\n")

# 1. 기준일 설정 (현재 날짜 기준)
base_date = date(2026, 7, 31)

# 2. Stage 3 파라미터 세팅
stage3_params = {
    'sga_lookback_quarters': 6,
    'require_sales_growth': True,
    'require_gpm_improvement': True,
    'require_inventory_turnover_up': True
}

try:
    loader = QuantDataLoader()
    screener = FundamentalImproveScreener(params=stage3_params)
    
    # 3. 테스트용 실제 종목 리스트 
    # (Stage 2를 통과했다고 가정. 예외 처리 작동 확인을 위해 금융주 추가)
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '005380', '105560'],  # 삼성전자, SK하이닉스, 현대차, KB금융
        'sector': ['IT', 'IT', '자동차', '금융'] 
    })
    
    print("⏳ DART API 실전 재무 데이터(최근 6개 분기 시계열) 수집 중...")
    print("(각 종목별로 과거 6분기치 데이터를 조합해야 하므로 시간이 소요될 수 있습니다.)\n")
    
    # 4. Stage 3 스크리너 실행
    passed_stage3_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 3 통과] 펀더멘털 개선(Turnaround) 확인 종목")
    display(passed_stage3_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date

from data.loader import QuantDataLoader
from stages.stage4_valuation import ValuationScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 4: 밸류에이션 (Valuation)")
print("==================================================\n")

# 1. 기준일 설정
base_date = date(2026, 7, 31)

# 2. 파라미터 설정
stage4_params = {
    'pbr_percentile_cutoff': 0.5,      # 섹터 내 PBR 하위 50%
    'bps_growth_yoy_min': 0.0,         # BPS 전년 대비 훼손 방지
    'roe_value_trap_threshold': 0.05   # ROE 5% 미만은 밸류트랩으로 간주
}

try:
    loader = QuantDataLoader()
    screener = ValuationScreener(params=stage4_params)
    
    # 3. 테스트용 종목 리스트 (Stage 3 통과 가정)
    # 4단계는 밸류트랩 검증을 위해 Stage 2에서 계산된 'roe' 값이 필수로 필요하므로 가상의 ROE를 주입합니다.
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '015760', '005380'],  # 삼성전자, SK하이닉스, 한국전력, 현대차
        'sector': ['IT', 'IT', '유틸리티', '자동차'],
        'roe': [0.12, 0.15, 0.01, 0.10]  # 한국전력(015760)을 밸류트랩으로 가정하여 ROE 1% 부여
    })
    
    print("⏳ pykrx 실전 펀더멘털 스냅샷(현재 및 1년 전) 수집 중...\n")
    
    # 4. Stage 4 스크리너 실행
    passed_stage4_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목 (ROE 포함)]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 4 통과] 밸류에이션 검증 완료 종목")
    display(passed_stage4_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from datetime import date

from data.loader import QuantDataLoader
from stages.stage5_financial_health import FinancialHealthScreener

print("==================================================")
print("🚀 [실전 테스트] Stage 5: 재무 건전성 (Financial Health)")
print("==================================================\n")

# 1. 기준일 설정
base_date = date(2026, 7, 31)

# 2. 파라미터 설정
stage5_params = {
    'debt_ratio_percentile_cutoff': 0.5,  # 섹터 내 부채비율 하위 50%
    'interest_coverage_min': 1.5          # 이자보상배율 1.5배 이상
}

try:
    loader = QuantDataLoader()
    screener = FinancialHealthScreener(params=stage5_params)
    
    # 3. 테스트용 실제 종목 리스트 (Stage 4 통과 가정)
    # 금융주 예외 처리 검증을 위해 KB금융(105560) 포함
    test_tickers_df = pd.DataFrame({
        'ticker': ['005930', '000660', '005380', '105560'],  # 삼성전자, SK하이닉스, 현대차, KB금융
        'sector': ['IT', 'IT', '자동차', '금융']
    })
    
    print("⏳ DART API TTM(최근 4개 분기 합산) 재무 데이터 수집 중...\n")
    
    # 4. Stage 5 스크리너 실행
    passed_stage5_df = screener.run(test_tickers_df, loader, base_date)
    
    print("📊 [입력된 테스트 종목]")
    display(test_tickers_df)
    
    print("\n✅ [Stage 5 통과] 최종 재무 건전성 검증 완료 종목")
    display(passed_stage5_df)
    
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

In [1]:
import pandas as pd
from datetime import date
import yaml

# 우리가 작성한 핵심 모듈 임포트
from data.loader import QuantDataLoader
from core.pipeline import QuantPipeline

print("==================================================")
print("🚀 [최종 통합 테스트] KOSPI 퀀트 파이프라인 전체 가동")
print("==================================================\n")

# 1. config/params.yaml 설정을 모사하는 전체 파라미터 딕셔너리
pipeline_params = {
    'stage1_neglected_sector': {
        'stage1_return_weight': 0.5,
        'stage1_volume_weight': 0.5,
        'stage1_pass_ratio': 0.3  # 하위 30% 소외 섹터만 통과
    },
    'stage2_sector_leaders': {
        'roe_percentile_cutoff': 0.5,
        'roic_percentile_cutoff': 0.5,
        'op_margin_std_percentile_cutoff': 0.5,
        'op_margin_lookback_q': 8,
        'op_margin_min_quarters': 4
    },
    'stage3_fundamental_improve': {
        'sga_lookback_quarters': 6,
        'require_sales_growth': True,
        'require_gpm_improvement': True,
        'require_inventory_turnover_up': True
    },
    'stage4_valuation': {
        'pbr_percentile_cutoff': 0.5,
        'bps_growth_yoy_min': 0.0,
        'roe_value_trap_threshold': 0.05
    },
    'stage5_financial_health': {
        'debt_ratio_percentile_cutoff': 0.5,
        'interest_coverage_min': 1.5
    }
}

# 2. 기준일 설정 (현재 날짜)
base_date = date(2026, 7, 31)

try:
    # 3. 로더 및 파이프라인 초기화
    loader = QuantDataLoader()
    pipeline = QuantPipeline(params=pipeline_params, loader=loader)
    
    print("⏳ 파이프라인 가동 중... (데이터 수집 및 분석에 시간이 소요됩니다.)\n")
    
    # 4. 파이프라인 실행 (최종 통과 종목 및 히스토리 반환)
    final_df, history = pipeline.run(base_date)
    
    # ---------------------------------------------------------
    # 5. 결과 및 단계별 탈락/통과 이력(History) 리포팅
    # ---------------------------------------------------------
    print("📈 [파이프라인 단계별 통과 현황]")
    
    if 'stage1' in history:
        print(f"✔️ Stage 1 (소외 섹터 선별) : {len(history['stage1'])}개 섹터 통과")
    if 'stage2' in history:
        print(f"✔️ Stage 2 (섹터 내 우량주) : {len(history['stage2'])}개 종목 통과")
    if 'stage3' in history:
        print(f"✔️ Stage 3 (체질 개선)     : {len(history['stage3'])}개 종목 통과")
    if 'stage4' in history:
        print(f"✔️ Stage 4 (밸류에이션)   : {len(history['stage4'])}개 종목 통과")
    if 'stage5' in history:
        print(f"✔️ Stage 5 (재무 건전성)   : {len(history['stage5'])}개 종목 최종 생존")
        
    print("\n==================================================")
    print("🏆 [최종 산출된 조기 은퇴 포트폴리오 후보군]")
    print("==================================================")
    
    if not final_df.empty:
        display(final_df)
    else:
        print("조건을 모두 만족하는 종목이 이번 달에는 없습니다. (시장 상황에 따라 자연스러운 현상일 수 있습니다.)")
        
    print("\n💡 Tip: 특정 단계에서 어떤 종목들이 떨어졌는지 분석하고 싶다면,")
    print("        `history['stage3']` 처럼 딕셔너리를 호출하여 확인할 수 있습니다.")

except Exception as e:
    print(f"❌ 파이프라인 실행 중 에러 발생: {e}")

KRX 로그인 시도...
  로그인 ID: forscom
KRX 로그인 완료.
  로그인 시간: 2026-08-02 11:13:27
  만료 시간: 2026-08-02 12:13:27
🚀 [최종 통합 테스트] KOSPI 퀀트 파이프라인 전체 가동

⏳ 파이프라인 가동 중... (데이터 수집 및 분석에 시간이 소요됩니다.)

reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
reprt_code='11

c:\Users\wwwss\quant_project\stock_screener\stages\stage2_sector_leaders.py:47: RuntimeWarning: divide by zero encountered in divide
  roic = np.divide(roic_num, roic_den)


reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개

c:\Users\wwwss\quant_project\stock_screener\stages\stage2_sector_leaders.py:47: RuntimeWarning: divide by zero encountered in divide
  roic = np.divide(roic_num, roic_den)


reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제

...

reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개

c:\Users\wwwss\quant_project\stock_screener\stages\stage2_sector_leaders.py:47: RuntimeWarning: divide by zero encountered in divide
  roic = np.divide(roic_num, roic_den)


reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
reprt_code='11013', fs_div='CFS' (1분기보고서, 연결제무제표)'
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개

c:\Users\wwwss\quant_project\stock_screener\stages\stage2_sector_leaders.py:47: RuntimeWarning: divide by zero encountered in divide
  roic = np.divide(roic_num, roic_den)


reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.

In [2]:
print("==================================================")
print("🔍 파이프라인 탈락률(Funnel) 분석")
print("==================================================\n")

# 각 단계별 생존 종목 수 확인
stages = ['stage1', 'stage2', 'stage3', 'stage4', 'stage5']
previous_count = None

for stage in stages:
    if stage in history:
        current_count = len(history[stage])
        
        # 탈락률 계산
        if previous_count is not None and previous_count > 0:
            drop_rate = (previous_count - current_count) / previous_count * 100
            print(f"📉 {stage} 생존: {current_count}개 (직전 단계 대비 -{drop_rate:.1f}% 탈락)")
        else:
            print(f"🎯 {stage} 생존: {current_count}개 (섹터 통과 후 남은 전체 유니버스)")
            
        previous_count = current_count

# 만약 Stage 4까지 살아남았는데 Stage 5에서 다 죽었다면, 
# Stage 4의 생존자들을 살펴봅니다.
if 'stage4' in history and not history['stage4'].empty:
    print("\n💡 [마지막 생존자] Stage 4 통과 종목 (이들이 Stage 5에서 탈락함)")
    display(history['stage4'][['ticker', 'sector', 'roe', 'pbr']])

🔍 파이프라인 탈락률(Funnel) 분석

🎯 stage1 생존: 37개 (섹터 통과 후 남은 전체 유니버스)
📉 stage2 생존: 75개 (직전 단계 대비 --102.7% 탈락)
📉 stage3 생존: 7개 (직전 단계 대비 -90.7% 탈락)
📉 stage4 생존: 1개 (직전 단계 대비 -85.7% 탈락)

💡 [마지막 생존자] Stage 4 통과 종목 (이들이 Stage 5에서 탈락함)


,ticker,sector,roe,pbr
0,100840,일반 목적용 기계 제조업,0.264711,1.35


In [3]:
import pandas as pd

print("==================================================")
print("🩺 [정밀 진단] Stage 3 탈락 종목 Raw Data 분석")
print("==================================================\n")

try:
    # 1. Stage 2 통과자 중 Stage 3에서 떨어진 종목 추출
    # history 딕셔너리에 데이터가 남아있다는 가정 하에 진행
    stage2_passed = history['stage2']
    stage3_passed_tickers = history['stage3']['ticker'].tolist() if not history['stage3'].empty else []
    
    stage3_failed_df = stage2_passed[~stage2_passed['ticker'].isin(stage3_passed_tickers)]
    sample_failures = stage3_failed_df.head(5)['ticker'].tolist()
    
    print(f"총 {len(stage3_failed_df)}개 종목이 Stage 3에서 탈락했습니다. 샘플 5개 진단 시작...\n")
    
    # 2. 로더를 직접 호출하여 데이터 계산 흐름 확인
    for ticker in sample_failures:
        print(f"▶️ 대상 종목: {ticker}")
        
        # 최근 6개 분기 시계열 로드
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        
        if len(q_series) < 6:
            print("  ❌ [사유] DATA_TOO_SHORT (가용 데이터 부족)\n")
            continue
            
        # 값 출력 (최근 3개 분기(t=0, 1, 2)와 전년 동기(t=4, 5, 6) 데이터 흐름 확인)
        for i, q in enumerate(q_series[:4]):  # 디버깅을 위해 최근 4분기만 출력
            rev = q.get('revenue', 0)
            sga = q.get('sga', 0)
            print(f"  [t-{i} 분기] 매출액: {rev:,.0f} | 판관비: {sga:,.0f}")
            
            if rev < 0 or sga < 0:
                print("  🚨 [경고] 음수 값 발견! 차분 로직(thstrm_add_amount) 오류 확실시 됨.")
                
        print("-" * 50)
        
except Exception as e:
    print(f"❌ 진단 중 에러 발생: {e}")

🩺 [정밀 진단] Stage 3 탈락 종목 Raw Data 분석

총 68개 종목이 Stage 3에서 탈락했습니다. 샘플 5개 진단 시작...

▶️ 대상 종목: 005380
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

  [t-0 분기] 매출액: 45,938,886,000,000 | 판관비: 5,501,853,000,000
  [t-1 분기] 매출액: 46,838,586,000,000 | 판관비: 6,148,060,000,000
  [t-2 분기] 매출액: 46,721,448,000,000 | 판관비: 5,746,793,000,000
  [t-3 분기] 매출액: 48,286,677,000,000 | 판관비: 5,508,233,000,000
--------------------------------------------------
▶️ 대상 종목:

In [5]:
import pandas as pd
from collections import Counter

print("==================================================")
print("🩺 Stage 3 진입 종목(75개) 전수 데이터 상태 판별 (로더 직접 호출)")
print("==================================================\n")

try:
    stage2_passed_df = history.get('stage2')
    if stage2_passed_df is None or stage2_passed_df.empty:
        print("Stage 2 통과 종목이 없습니다.")
    else:
        status_counter = Counter()
        
        print("⏳ 로더(loader)를 통해 75개 종목의 시계열 상태 직접 확인 중...")
        for _, row in stage2_passed_df.iterrows():
            ticker = row['ticker']
            
            # loader를 직접 호출하여 6분기 시계열 확보 시도 (에러 없이 가져오는지 확인)
            q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
            
            if len(q_series) < 6:
                status_counter['DATA_TOO_SHORT (6분기 미달)'] += 1
            else:
                # 데이터가 6개 다 있다면, 음수나 결측치가 있는지 추가 확인
                has_error = False
                for q in q_series:
                    rev = q.get('revenue', 0)
                    sga = q.get('sga', 0)
                    # 데이터가 없거나(NaN), 음수값이 튀어나온 경우(단독값 파싱 에러 의심)
                    if pd.isna(rev) or pd.isna(sga) or rev < 0 or sga < 0:
                        has_error = True
                        break
                        
                if has_error:
                    status_counter['DATA_INVALID (음수 또는 결측치 포함)'] += 1
                else:
                    status_counter['COMPUTED (정상 계산 가능)'] += 1
            
        print("\n📊 [전수조사 결과: 데이터 상태 분포]")
        total = len(stage2_passed_df)
        for status, count in status_counter.items():
            print(f" - {status}: {count}개 종목 ({(count/total)*100:.1f}%)")

except Exception as e:
    print(f"❌ 진단 중 에러 발생: {e}")

🩺 Stage 3 진입 종목(75개) 전수 데이터 상태 판별 (로더 직접 호출)

⏳ 로더(loader)를 통해 75개 종목의 시계열 상태 직접 확인 중...
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_cod

In [6]:
import pandas as pd

print("==================================================")
print("🕵️‍♂️ [심층 추적] 14개 비정상 데이터 원본 파싱 상태 점검")
print("==================================================\n")

try:
    stage2_passed_df = history['stage2']
    invalid_tickers = []
    
    for _, row in stage2_passed_df.iterrows():
        ticker = row['ticker']
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        
        # 6분기 데이터가 모두 존재하는 경우에만 값 검증
        if len(q_series) == 6:
            for q in q_series:
                rev = q.get('revenue', 0)
                sga = q.get('sga', 0)
                # 성장률(변동량)이 아닌, '매출액/판관비 절대치' 자체가 음수인 경우를 적발
                if pd.isna(rev) or pd.isna(sga) or rev < 0 or sga < 0:
                    invalid_tickers.append(ticker)
                    break
    
    print(f"발견된 의심 종목 수: {len(invalid_tickers)}개\n")
    
    # 샘플 3개만 상세 출력하여 원본 값 확인
    for ticker in invalid_tickers[:3]:
        print(f"▶️ 종목: {ticker}")
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        for i, q in enumerate(q_series):
            rev_val = q.get('revenue')
            sga_val = q.get('sga')
            
            rev_str = f"{rev_val:>20,}" if pd.notna(rev_val) else f"{'NaN':>20}"
            sga_str = f"{sga_val:>20,}" if pd.notna(sga_val) else f"{'NaN':>20}"
            
            print(f"  [t-{i}] 매출액: {rev_str} | 판관비: {sga_str}")
        print("-" * 50)
        
except Exception as e:
    print(f"❌ 에러: {e}")

🕵️‍♂️ [심층 추적] 14개 비정상 데이터 원본 파싱 상태 점검

reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'sta

In [19]:
import pandas as pd

print("==================================================")
print("🛠️ [패치 검증] 로더 파싱 로직 수정 후 데이터 확인")
print("==================================================\n")

# 아까 결측치(NaN) 릴레이와 4분기 누락 문제가 발생했던 타겟 종목들
# 103140(풍산), 069620(대동), 003090(대웅)
test_tickers = ['103140', '069620', '003090']

try:
    for ticker in test_tickers:
        print(f"▶️ 테스트 종목: {ticker}")
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        
        for i, q in enumerate(q_series):
            rev_val = q.get('revenue')
            sga_val = q.get('sga')
            
            # 보기 편하게 포맷팅
            rev_str = f"{rev_val:>20,}" if pd.notna(rev_val) else f"{'NaN':>20}"
            sga_str = f"{sga_val:>20,}" if pd.notna(sga_val) else f"{'NaN':>20}"
            
            print(f"  [t-{i}] 매출액: {rev_str} | 판관비: {sga_str}")
        print("-" * 50)
        
    print("✅ 테스트 완료! 결과를 확인해 보세요.")

except Exception as e:
    print(f"❌ 에러 발생: {e}")

🛠️ [패치 검증] 로더 파싱 로직 수정 후 데이터 확인

▶️ 테스트 종목: 103140
reprt_code='11014', fs_div='CFS' (3분기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11014', fs_div='OFS' (3분기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='CFS' (반기보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11012', fs_div='OFS' (반기보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

reprt_code='11011', fs_div='OFS' (사업보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

  [t-0] 매출액:  1,270,895,673,812.0 | 판관비:                  NaN
  [t-1] 매출액:                  NaN | 판관비:                  NaN
  [t-2] 매출액:  1,174,161

In [17]:
import importlib

# 1. 현재 loader 객체가 속한 모듈(파일)을 동적으로 찾아 새로고침
loader_module = __import__(loader.__module__, fromlist=[''])
importlib.reload(loader_module)

# 2. 기존 loader와 완벽히 동일한 클래스로 새 인스턴스 생성
LoaderClass = type(loader)
# 주의: 만약 처음에 loader = LoaderClass(api_key="...") 처럼 인자를 넣으셨다면 
# 아래 괄호 안에도 동일하게 넣어주셔야 합니다.
loader = LoaderClass() 

print(f"✅ {LoaderClass.__name__} 모듈 업데이트 완료!")

✅ QuantDataLoader 모듈 업데이트 완료!
